# Report

Comprehensive description of the pipeline, algorithm, and complexity analysis.

## Problem Statement

Given a set of source code files, identify pairs with high structural or token-level similarity to detect likely code reuse or plagiarism.

The result must be is a ranked list of file pairs and flagged pairs above a configured threshold.

## Approach / Paradigm

- **Main paradigm:** **Dynamic programming** (for Damerau-Levenshtein distance
- **Justification:** The core similarity measure is an edit-distance that admits an optimal substructure and overlapping subproblems. DP yields an exact minimum edit sequence cost.

## Algorithm Steps

### Top Level

```
FUNCTION analyze(files, threshold):
    FOR each file:
        parse file into AST
        normalize AST (replace identifiers, literals with placeholders)
        split AST into chunks (one per top-level statement or logical block)
        convert each chunk into a token sequence

    FOR each unique pair of files (A, B):
        score = compare_files(A, B)
        store score

    sort scores from highest to lowest
    flag pairs where score >= threshold

    RETURN report
```

---

### Comparing Two Files

```
FUNCTION compare_files(A, B):
    forward  = best_match_score(A, B)   // how well A is covered by B
    backward = best_match_score(B, A)   // how well B is covered by A

    RETURN average(forward, backward)
```

---

### Best Match Score (One Direction)

```
FUNCTION best_match_score(primary, secondary):
    total_weight = 0
    weighted_sum = 0

    FOR each chunk P in primary:
        best_similarity = 0

        FOR each chunk S in secondary:
            similarity = chunk_similarity(P, S)

            IF similarity > best_similarity:
                best_similarity = similarity

        weight = length of the longer token sequence (P or S)
        weighted_sum += best_similarity × weight
        total_weight += weight

    RETURN weighted_sum / total_weight
```

---

### Chunk Similarity

```
FUNCTION chunk_similarity(P, S):
    distance = damerau_levenshtein(P.tokens, S.tokens)
    max_len  = max(len(P.tokens), len(S.tokens))

    IF max_len == 0:
        RETURN 1.0              // both empty, identical

    RETURN 1.0 - (distance / max_len)
```

---

### Damerau-Levenshtein Distance (Core DP)

```
FUNCTION damerau_levenshtein(A, B):
    IF A == B:
        RETURN 0  // identical, no work needed

    IF A is empty: RETURN len(B)
    IF B is empty: RETURN len(A)

    // allocate DP matrix with two extra rows/cols for sentinel handling
    CREATE matrix of size (len(A)+2) × (len(B)+2)

    matrix[0][0] = len(A) + len(B)  // large sentinel
    FOR i from 0 to len(A):
        matrix[i+1][0] = large sentinel
        matrix[i+1][1] = i
    FOR j from 0 to len(B):
        matrix[0][j+1] = large sentinel
        matrix[1][j+1] = j

    last_seen = {}

    FOR each token A[i] in A:
        last_match = 0

        FOR each token B[j] in B:
            prev_i = last_seen[B[j]] or 0
            prev_j = last_match

            IF A[i] == B[j]:
                cost = 0
                last_match = j
            ELSE:
                cost = 1

            substitute = matrix[i][j]   + cost
            insert     = matrix[i+1][j] + 1
            delete     = matrix[i][j+1] + 1
            transpose  = matrix[prev_i][prev_j] + (i - prev_i - 1) + 1 + (j - prev_j - 1)

            matrix[i+1][j+1] = MIN(substitute, insert, delete, transpose)

        last_seen[A[i]] = i

    RETURN matrix[len(A)+1][len(B)+1]
```

## Complexity Analysis

**Let:**
- $n$: number of files (submissions)
- $k$: average number of chunks per file
- $t$: average number of tokens per chunk

### Breakdown

**Parsing & Chunking**
- Per file: $O(k t)$ - traverse AST and produce tokens once.
- All files: $O(n k t)$.
- Best = Average = Worst = $O(n k t)$.

**Damerau-Levenshtein Distance (DP)**
- For two token sequences lengths $t_1$ and $t_2$, DP time is $O(t_1 t_2)$.
- Using average $t$ gives $O(t^2)$ per chunk pair.

**Chunk Alignment (one file pair)**
- For each chunk in $A$ compare to every chunk in $B$: $O(k^2 t^2)$.
- The directional `best_match_score` runs twice ($A \to B$ and $B \to A$), but the constant factor 2 does not change big-O.

**All-Pairs File Comparison**
- There are $O(n^2)$ file pairs.
- Total time: $O(n^2 k^2 t^2)$.

**Sorting / Reporting**
- Sorting $O(n^2 \log n)$ is dominated by the $O(n^2 k^2 t^2)$ term.

### Total 
- Overall: $T(n, k, t) = O(n^2 k^2 t^2)$.

### Case-by-case

- **Best:** If many pairs are exactly identical, the DL function may short-circuit on equality checks reducing constant factors, but structural loops remain: $O(n^2 k^2 t^2)$

- **Average:** $O(n^2 k^2 t^2)$ for typical submissions.

- **Worst:** Distinct files with full-length chunks: $O(n^2 k^2 t^2)$.

### Space Complexity
- Token storage for all files: $O(n k t)$.
- DP matrix for one comparison (reused): $O(t^2)$.
- Similarity scores structure: $O(n^2)$.

**Total space:** $O(n k t + t^2 + n^2)$

## Limitations

- The $O(n^2 k^2 t^2)$ complexity limits scaling for large $n$ or very large chunks. For classroom-sized datasets ($n \approx 10-100$ with minimal content), this is usually acceptable.


## Possible Improvements

- Use locality-sensitive hashing (LSH) or MinHash on token n-grams to prefilter candidate file pairs, reducing effective $n$.
- Apply chunk-level pruning: compare only chunks whose length ratio falls within a threshold, or use signature sketches to avoid full DP for low-probability matches.
- Parallelize pairwise comparisons across CPU cores or workers since DP comparisons are independent.
- Cache previously computed chunk-pair distances if identical chunks recur across files.